# DiariZen-Large — standalone runner

DiariZen cannot share an environment with the main notebook: it pins
`numpy==1.26.4`, while `pyannote.audio` 4.x pulls numpy 2.5.x. Rather than fight
that, it runs **here**, in its own session, and exports plain RTTM files that the
main pipeline adopts with `diarization.import_external_rttm()`.

That keeps one scoring implementation. DiariZen is scored, ranked and shown in
the error explorer exactly like the in-process models — the only difference is
where its turns came from.

**Kaggle settings:** Internet **ON**, Accelerator **GPU**, and attach the same
audio dataset the main notebook uses.

Model: `BUT-FIT/diarizen-wavlm-large-s80-md-v2` — not gated, CC-BY-NC-4.0
(non-commercial; fine for evaluation, check before any commercial use).

In [ ]:
# DiariZen is GitHub-only (not on PyPI) and pins numpy==1.26.4.
# Install it ALONE in this session -- do NOT add pyannote.audio 4.x here, that
# numpy pin is exactly why this notebook is separate.
import subprocess, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
SRC = Path("/kaggle/working/DiariZen") if IN_KAGGLE else Path("./DiariZen")

if not SRC.exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/BUTSpeechFIT/DiariZen.git", str(SRC)], check=True)
    print("cloned", SRC)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SRC)], check=True)

# That install downgrades numpy, so the kernel almost certainly needs a restart
# before diarizen is importable -- same stale-interpreter check as the main
# notebook, for the same reason.
_loaded = __import__("numpy").__version__
_disk = subprocess.run([sys.executable, "-c", "import numpy;print(numpy.__version__)"],
                       capture_output=True, text=True).stdout.strip()
if _disk and _disk != _loaded:
    print("=" * 68)
    print("RESTART THE KERNEL, then run this cell again.")
    print(f"  numpy: loaded {_loaded}, on disk {_disk}")
    print("  Run > Restart & clear cell outputs")
    print("=" * 68)
    raise SystemExit("restart required")
print("numpy", _loaded, "-- ready")

## Paths

`AUDIO_DIR` is the same audio the main notebook uses. `OUT_DIR` receives one
`<clip_id>.rttm` per clip — that folder is the entire handoff.

In [ ]:
from pathlib import Path
import os

IN_KAGGLE = Path("/kaggle").exists()

# Same dataset the main notebook uses -- check the Input panel for the slug.
AUDIO_DIR = Path("/kaggle/input/dataset/audio_16k") if IN_KAGGLE else Path("local_out/audio_16k")
OUT_DIR   = Path("/kaggle/working/diarizen_rttm") if IN_KAGGLE else Path("./diarizen_rttm")
OUT_DIR.mkdir(parents=True, exist_ok=True)

wavs = sorted(AUDIO_DIR.glob("*.wav"))
assert wavs, f"no wavs under {AUDIO_DIR} -- attach the audio dataset and fix AUDIO_DIR"
print(f"{len(wavs)} clips  ->  {OUT_DIR}")

In [ ]:
import time, torch
from diarizen.pipelines.inference import DiariZenPipeline

MODEL = "BUT-FIT/diarizen-wavlm-large-s80-md-v2"
pipe = DiariZenPipeline.from_pretrained(MODEL)
if torch.cuda.is_available():
    pipe.to(torch.device("cuda"))
print("loaded", MODEL, "on", "cuda" if torch.cuda.is_available() else "cpu")

def to_rttm(clip_id, ann):
    """Same RTTM shape the main pipeline writes: start + DURATION, no spaces
    in the speaker label."""
    out = []
    for seg, _, label in ann.itertracks(yield_label=True):
        if seg.end <= seg.start:
            continue
        spk = "_".join(str(label).split())
        out.append(f"SPEAKER {clip_id} 1 {seg.start:.3f} {seg.end - seg.start:.3f} "
                   f"<NA> <NA> {spk} <NA> <NA>")
    return "\n".join(out) + ("\n" if out else "")

done = skipped = failed = 0
t0 = time.time()
for i, wav in enumerate(wavs, 1):
    dst = OUT_DIR / f"{wav.stem}.rttm"
    if dst.exists():                      # resumable, like every other stage
        skipped += 1
        continue
    try:
        t = time.time()
        ann = pipe(str(wav))
        dst.write_text(to_rttm(wav.stem, ann), encoding="utf-8")
        done += 1
        print(f"[{i}/{len(wavs)}] {wav.stem}  {time.time()-t:.1f}s")
    except Exception as exc:
        failed += 1
        print(f"[{i}/{len(wavs)}] {wav.stem}  FAILED {type(exc).__name__}: {exc}")

print(f"\n{done} written, {skipped} already present, {failed} failed "
      f"in {(time.time()-t0)/60:.1f} min")

## Hand off to the main pipeline

Zip the RTTMs and download them (Output tab). Then in the **main** notebook,
after cell 2.2:

```python
from sarvam_diar import diarization
diarization.import_external_rttm(
    cfg,
    "/kaggle/input/<diarizen-rttm-dataset>",   # or a local path
    model="diarizen-large",
    clip_durations={c.clip_id: c.duration for c in inputs.values()},
)
```

From then on `diarizen-large` is scored, ranked and rendered in the error
explorer exactly like the in-process models. Nothing else changes.

In [ ]:
import shutil, os
zip_path = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR)
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB, "
      f"{len(list(OUT_DIR.glob('*.rttm')))} rttm files)")
print("\nDownload from the Output tab, or Save Version and attach it to the")
print("main notebook as an Input.")